# 🏥 Hospital Resource Optimisation with QAOA
### Quantum for Healthcare — Quantum for Humanity

This notebook applies **QAOA** to optimise hospital bed allocation and staff scheduling — maximising patient throughput with limited healthcare resources.

**Application:** Applicable to rural hospitals, pandemic surge planning, and disaster response.  
**Learning source:** [IBM Quantum Learning](https://learning.quantum.ibm.com)

---

## Clinical Motivation

In low-resource healthcare settings, efficient resource allocation is the difference between life and death:
- **Bed allocation** across departments (ICU, General, Paediatric, Maternity)
- **Surgical scheduling** to minimise patient wait times
- **Staff rostering** to match patient demand
- **Ambulance routing** for emergency response

These are all **combinatorial optimisation problems** — ideal for QAOA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Sampler

print('✅ Imports successful')

## Step 1: Define Hospital Resource Problem

A district hospital has **8 patient cases** awaiting treatment and limited ICU beds. We need to optimise which patients get ICU admission to maximise clinical outcomes.

In [ ]:
# Patient cases awaiting ICU admission
patients = [
    # (id, condition,        severity, survival_benefit, icu_days, age)
    ('P01', 'Septic Shock',       9.5,   0.92,  5,  42),
    ('P02', 'Cardiac Arrest',     9.8,   0.88,  4,  67),
    ('P03', 'Trauma (MVA)',       8.5,   0.85,  7,  28),
    ('P04', 'Respiratory Fail.',  8.0,   0.80,  6,  55),
    ('P05', 'Post-Op Complic.',   7.5,   0.78,  3,  73),
    ('P06', 'Eclampsia',          9.0,   0.91,  4,  31),   # maternal
    ('P07', 'Paediatric Fever',   7.0,   0.95,  2,   4),   # child
    ('P08', 'Diabetic Ketoacid.', 6.5,   0.75,  3,  61),
]

ids      = [p[0] for p in patients]
severity = np.array([p[2] for p in patients])
benefit  = np.array([p[3] for p in patients])  # P(survival | ICU admission)
icu_days = np.array([p[4] for p in patients])  # bed-days needed

ICU_BEDS     = 3    # total ICU beds available
MAX_BED_DAYS = 15   # total bed-days capacity this week

print('Patients Awaiting ICU:')
print(f'{"ID":<5} {"Condition":<22} {"Severity":>9} {"Benefit":>8} {"ICU Days":>9}')
print('-' * 60)
for p in patients:
    print(f'{p[0]:<5} {p[1]:<22} {p[2]:>9.1f} {p[3]:>8.2f} {p[4]:>9}')
print(f'\nICU Capacity: {ICU_BEDS} beds | {MAX_BED_DAYS} bed-days/week')

## Step 2: Build the Optimisation Problem (QUBO)

In [ ]:
n = len(patients)
qp = QuadraticProgram(name='ICU_Allocation')

# Binary variable: x_i = 1 if patient i is admitted to ICU
for pid in ids:
    qp.binary_var(name=pid)

# Objective: maximise total clinical benefit (weighted by severity)
objective = {ids[i]: -(severity[i] * benefit[i]) for i in range(n)}
qp.minimize(linear=objective)

# Constraint 1: number of patients ≤ ICU_BEDS
qp.linear_constraint(
    linear={ids[i]: 1 for i in range(n)},
    sense='<=', rhs=ICU_BEDS, name='bed_count'
)

# Constraint 2: total bed-days ≤ MAX_BED_DAYS
qp.linear_constraint(
    linear={ids[i]: int(icu_days[i]) for i in range(n)},
    sense='<=', rhs=MAX_BED_DAYS, name='bed_days'
)

print(qp.export_as_lp_string())

## Step 3: Classical Optimal Allocation

In [ ]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact.solve(qp)

print('\n📊 Classical Optimal ICU Admission:')
admitted = []
for i, x in enumerate(exact_result.x):
    if x > 0.5:
        p = patients[i]
        print(f'  ✅ Admit {p[0]}: {p[1]:<22} | Severity: {p[2]:.1f} | Benefit: {p[3]:.2f} | Days: {p[4]}')
        admitted.append(i)

total_benefit = sum(severity[i]*benefit[i] for i in admitted)
total_days    = sum(icu_days[i] for i in admitted)
print(f'\n  Total patients admitted: {len(admitted)} / {ICU_BEDS} beds')
print(f'  Total bed-days: {total_days} / {MAX_BED_DAYS}')
print(f'  Total clinical value: {total_benefit:.2f}')

## Step 4: Quantum QAOA Solution

In [ ]:
# Convert to QUBO
converter = QuadraticProgramToQubo()
qubo = converter.convert(qp)

# Run QAOA
qaoa = QAOA(sampler=Sampler(), optimizer=COBYLA(maxiter=300), reps=3)
qaoa_result_qubo = MinimumEigenOptimizer(qaoa).solve(qubo)
qaoa_result = converter.interpret(qaoa_result_qubo)

print('\n⚛️  QAOA Quantum ICU Allocation:')
admitted_q = []
for i, x in enumerate(qaoa_result.x):
    if x > 0.5:
        p = patients[i]
        print(f'  ⚛️  Admit {p[0]}: {p[1]:<22} | Severity: {p[2]:.1f} | Benefit: {p[3]:.2f} | Days: {p[4]}')
        admitted_q.append(i)

total_benefit_q = sum(severity[i]*benefit[i] for i in admitted_q)
total_days_q    = sum(icu_days[i] for i in admitted_q)
print(f'\n  Total patients admitted: {len(admitted_q)} / {ICU_BEDS} beds')
print(f'  Total bed-days: {total_days_q} / {MAX_BED_DAYS}')
print(f'  Total clinical value: {total_benefit_q:.2f}')

## Step 5: Visualise the Allocation Decision

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = []
for i in range(n):
    if i in admitted_q:
        colors.append('#8B5CF6')   # QAOA admitted
    elif i in admitted:
        colors.append('#FF9933')   # Classical only
    else:
        colors.append('#D1D5DB')   # Not admitted

x_pos = np.arange(n)
bars = ax.bar(x_pos, severity * benefit, color=colors, edgecolor='white', linewidth=1.5)

ax.set_xticks(x_pos)
ax.set_xticklabels([f"{p[0]}\n{p[1].split(' ')[0]}" for p in patients], fontsize=9)
ax.set_ylabel('Clinical Value (Severity × Benefit)', fontsize=11)
ax.set_title('ICU Allocation — Quantum QAOA vs Classical\n(Maximise clinical value within capacity constraints)',
             fontsize=12, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#8B5CF6', label='Admitted (QAOA ⚛️)'),
    Patch(facecolor='#FF9933', label='Classical only'),
    Patch(facecolor='#D1D5DB', label='Not admitted')
]
ax.legend(handles=legend_elements)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 🌍 Humanitarian Applications at Scale

This quantum optimisation framework can be applied to:

| Problem | Scale | Impact |
|---|---|---|
| **Pandemic surge ICU allocation** | 1000s of patients | COVID-19, future pandemics |
| **Surgical scheduling** | 100s of operations/week | Waiting list reduction |
| **Ambulance routing** | 10s-100s of vehicles | Response time → survival |
| **Vaccine distribution** | Millions of doses | Disease elimination |
| **Blood supply chain** | National blood banks | Zero shortage |
| **Rural health worker deployment** | 1000s of workers | Last-mile healthcare |

> *"Quantum optimisation applied to healthcare logistics could save tens of thousands of lives every year — not by inventing new medicine, but by ensuring the right care reaches the right person at the right time."*  
> — Quantum for Humanity

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*